In [1]:
import numpy as np                        # ndarrys for gridded data
import pandas as pd                       # DataFrames for tabular data
import os                                 # set working directory, run executables
import matplotlib.pyplot as plt           # for plotting
import matplotlib
from scipy import stats                   # summary statistics
import math                               # trig etc.
import scipy.signal as signal             # kernel for moving window calculation
import random                             # random sampling
from scipy.stats import gaussian_kde      # for PDF calculation
from scipy.stats import t                 # Student's t distribution for analytical solution
import seaborn as sns                     # for advanced plotting

In [5]:
def custom_histogram(zdata,zmin,zmax,xlabel,title):
    offset = (zmax-zmin)/30.0
    freq = plt.hist(zdata,color = 'red',alpha = 0.3,edgecolor='black',bins = np.linspace(zmin,zmax,int(len(zdata)/3)))[0]
    ht = np.max(freq)
    plt.axvline(x=np.average(zdata),linestyle="--",c='black')
    plt.text(np.average(zdata)+offset,ht*0.95, r'Average = ' + str(round(np.average(zdata),1)), fontsize=12)
    plt.text(np.average(zdata)+offset,ht*0.90, r'St.Dev. = ' + str(round(np.std(zdata),1)), fontsize=12)
    plt.text(np.average(zdata)+offset,ht*0.85, r'P10 = ' + str(round(np.percentile(zdata,10),1)), fontsize=12)
    plt.text(np.average(zdata)+offset,ht*0.80, r'P90 = ' + str(round(np.percentile(zdata,90),1)), fontsize=12)
    plt.xlabel(xlabel); plt.ylabel('Frequency'); plt.title(title)
     
def custom_histogram_with_uncert(zreal,zmin,zmax,zdata,xlabel,title):
    offset = (zmax-zmin)/30.0
    freq = plt.hist(zdata,color = 'red',alpha = 0.2,edgecolor='grey',bins = np.linspace(zmin,zmax,int(len(zdata)/3)))[0]
    ht = np.max(freq)
    plt.axvline(x=np.average(zdata),c='black')
    plt.axvline(x=np.percentile(zreal,90),linestyle="--",c='black')
    plt.axvline(x=np.percentile(zreal,10),linestyle="--",c='black')
    plt.text(np.percentile(zreal,90)+offset,ht*0.90, r'Average = ' + str(round(np.average(zreal),1)), fontsize=12)
    plt.text(np.percentile(zreal,90)+offset,ht*0.85, r'Average P90 = ' + str(round(np.percentile(zreal,90),1)), fontsize=12)
    plt.text(np.percentile(zreal,90)+offset,ht*0.80, r'Average P10 = ' + str(round(np.percentile(zreal,10),1)), fontsize=12)
    plt.xlabel(xlabel); plt.ylabel('Frequency'); plt.title(title)

def display_bootstrap(zreal,zmin,zmax,zdata,title,abin,analytical,stat):
    offset = (zmax-zmin)/30.0
    freq = plt.hist(zreal,color = 'red',alpha = 0.2,edgecolor = 'black',bins=np.linspace(zmin,zmax,100),label = 'bootstrap',density = True)[0] # plot the distribution, could also calculate any summary statistics
    ht = np.max(freq)
    if analytical is None:
        sns.kdeplot(x=zreal,color = 'grey',alpha = 0.1,levels = 1,bw_adjust = 1,label='bootstrap PDF') 
        plt.xlabel('Boostrap Realizations and Kernel Density Estimate') 
    else:    
        plt.plot(abin,analytical,color = 'black',label = 'analytical',alpha=0.4)
        plt.fill_between(abin, 0, analytical, where = abin <= np.percentile(zreal,10), facecolor='red', interpolate=True, alpha = 0.5)
        plt.fill_between(abin, 0, analytical, where = abin >= np.percentile(zreal,90), facecolor='red', interpolate=True, alpha = 0.5)
        plt.xlabel('Boostrap Realizations and Analytical Sampling Distributions') 
    plt.axvline(x=stat(zdata),linestyle="--",c='black')
    plt.text(stat(zdata)+offset,ht*0.95, r'Average = ' + str(round(np.average(zreal),1)), fontsize=12)
    plt.text(stat(zdata)+offset,ht*0.90, r'St.Dev. = ' + str(round(np.std(zreal),1)), fontsize=12)
    plt.text(stat(zdata)+offset,ht*0.85, r'P90 = ' + str(round(np.percentile(zreal,90),1)), fontsize=12)
    plt.text(stat(zdata)+offset,ht*0.8, r'P10 = ' + str(round(np.percentile(zreal,10),1)), fontsize=12)
    plt.ylabel('Frequency'); plt.title(title)
    plt.legend(loc = 'upper left')

In [6]:
seed = 13                                 # random number generator seed 
random.seed(a=seed)                       # initialize the random number generator

In [7]:
def bootstrap(zdata,nreal,stat):
    zreal = []                            # declare an empty list to store the bootstrap realizations
    for l in range(0,nreal):              # loop over the L bootstrap realizations
        samples = random.choices(zdata, k=len(zdata)) # n Monte Carlo simulations, sample with replacement
        zreal.append(stat(samples))       # calculate the realization of the statistic and append to list
    return zreal                          # return the list of realizations of the statistic